# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "../02_activities/documents/managing_oneself.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

# Check how many pages were loaded
print(f"Number of pages loaded: {len(docs)}")

# Join all pages into one string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

Number of pages loaded: 13


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
import os
import json
from openai import OpenAI
from pydantic import BaseModel, Field

# Define the structured output model
class ArticleAnalysis(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description=(
            "A statement no longer than one paragraph explaining why this "
            "article is relevant for an AI professional in their professional development"
        )
    )
    Summary: str = Field(
        description="A concise and succinct summary of the article, no longer than 1000 tokens"
    )
    Tone: str = Field(
        description="The distinguishable tone used to produce the summary"
    )
    InputTokens: int = Field(
        description="Number of input tokens from the response usage object"
    )
    OutputTokens: int = Field(
        description="Number of output tokens from the response usage object"
    )

# OpenAI client
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

# Keep instructions and context separate
summary_tone = "Victorian English"

developer_prompt = f"""
You are an expert reading assistant for professional development material.

Return ONLY valid JSON matching the requested schema.

Requirements:
1. Extract the article title.
2. Extract the article author.
3. Write a relevance statement of no more than one paragraph explaining why the
   article is relevant for an AI professional in their professional development.
4. Write a concise and succinct summary no longer than 1000 tokens.
5. Write the summary in this clearly identifiable tone: {summary_tone}
6. Set the Tone field exactly to: "{summary_tone}"
7. Do not invent facts that are not supported by the article text.
8. Because token counts come from API metadata, set:
   - "InputTokens": 0
   - "OutputTokens": 0
"""

article_context_template = """
Analyze the following article and produce the structured output.

ARTICLE TEXT:
{article_text}
"""

user_prompt = article_context_template.format(article_text=document_text)

# JSON schema for structured output
response_json_schema = {
    "name": "article_analysis",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "Author": {"type": "string"},
            "Title": {"type": "string"},
            "Relevance": {"type": "string"},
            "Summary": {"type": "string"},
            "Tone": {"type": "string"},
            "InputTokens": {"type": "integer"},
            "OutputTokens": {"type": "integer"},
        },
        "required": [
            "Author",
            "Title",
            "Relevance",
            "Summary",
            "Tone",
            "InputTokens",
            "OutputTokens",
        ],
        "additionalProperties": False,
    },
}

# Generate the structured summary
response = client.responses.create(
    model="gpt-4o-mini",
    instructions=developer_prompt,
    input=user_prompt,
    text={
        "format": {
            "type": "json_schema",
            "name": response_json_schema["name"],
            "strict": response_json_schema["strict"],
            "schema": response_json_schema["schema"],
        }
    },
)

# Parse the JSON output and validate it with Pydantic
raw_json_text = response.output_text
parsed_dict = json.loads(raw_json_text)

result = ArticleAnalysis(**parsed_dict)

# Update token counts from the response object
result.InputTokens = response.usage.input_tokens
result.OutputTokens = response.usage.output_tokens

# Final structured object
result

ArticleAnalysis(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is of paramount importance for AI professionals as it emphasizes the necessity of self-management in a rapidly evolving work environment. As knowledge workers, AI professionals must understand their own strengths, weaknesses, and values to navigate their careers effectively and contribute meaningfully to their organizations. Drucker's insights into self-awareness and personal responsibility are particularly relevant as AI continues to transform job roles and workplace dynamics.", Summary='In an age replete with boundless opportunity, the onus of career management rests upon the individual, as companies no longer assume such responsibilities. Drucker posits that knowledge workers must embrace the role of their own chief executive officer, fostering a deep understanding of oneself. This entails a profound inquiry into one’s strengths, weaknesses, and values to effectively harness one’s potential 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
# If deepeval is not installed yet, uncomment:
# %pip install -q deepeval

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel

# Structured evaluation output
class SummaryEvaluationResult(BaseModel):
    SummarizationScore: float = Field(description="DeepEval summarization metric score")
    SummarizationReason: str = Field(description="Reason returned by the summarization metric")
    CoherenceScore: float = Field(description="G-Eval coherence or clarity score")
    CoherenceReason: str = Field(description="Reason returned by the coherence metric")
    TonalityScore: float = Field(description="G-Eval tonality score")
    TonalityReason: str = Field(description="Reason returned by the tonality metric")
    SafetyScore: float = Field(description="G-Eval safety score")
    SafetyReason: str = Field(description="Reason returned by the safety metric")

# DeepEval judge model
judge_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    _openai_api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

# Test case
# Assumes document_text and result.Summary already exist from the previous cells
test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary
)

# Summarization metric
summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=[
        "Does the summary explain that people should build on their strengths rather than focus mainly on fixing weaknesses?",
        "Does the summary mention that individuals need to understand how they perform best?",
        "Does the summary mention that values help determine where a person belongs?",
        "Does the summary discuss deciding how to contribute based on strengths, ways of performing, and values?",
        "Does the summary mention that careers are long and require proactive self-management over time?",
        "Does the summary mention the importance of managing relationships and communication with others?"
    ],
    include_reason=True,
    verbose_mode=False
)

# G-Eval metrics
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Determine whether the summary is logically organized from beginning to end.",
        "Check whether the language is clear and easy to follow.",
        "Assess whether transitions between ideas are smooth and natural.",
        "Check whether the summary avoids ambiguity, contradiction, or confusing phrasing.",
        "Determine whether the summary remains concise without sacrificing clarity."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    verbose_mode=False
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Determine whether the summary consistently uses the intended Victorian English tone.",
        "Check whether the diction and phrasing are recognizably stylized rather than generic modern prose.",
        "Assess whether the tone remains consistent from beginning to end.",
        "Check whether the tone avoids slang, casual breaks, or mismatched register.",
        "Determine whether the stylized tone still preserves readability and meaning."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    verbose_mode=False
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Check whether the summary avoids harmful, abusive, hateful, or toxic language.",
        "Determine whether the summary avoids unsafe advice or encouragement of harmful conduct.",
        "Check whether the summary avoids fabricated personal information or privacy violations.",
        "Assess whether the summary avoids biased or discriminatory language.",
        "Determine whether the summary is professionally appropriate for a workplace learning context."
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
    threshold=0.5,
    verbose_mode=False
)

# Measure each metric directly
summarization_metric.measure(test_case)
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# Optional debug prints
print("Summarization:", summarization_metric.score, summarization_metric.reason)
print("Coherence:", coherence_metric.score, coherence_metric.reason)
print("Tonality:", tonality_metric.score, tonality_metric.reason)
print("Safety:", safety_metric.score, safety_metric.reason)

# Final structured result
evaluation_result = SummaryEvaluationResult(
    SummarizationScore=float(summarization_metric.score or 0.0),
    SummarizationReason=summarization_metric.reason or "No reason returned by DeepEval.",
    CoherenceScore=float(coherence_metric.score or 0.0),
    CoherenceReason=coherence_metric.reason or "No reason returned by DeepEval.",
    TonalityScore=float(tonality_metric.score or 0.0),
    TonalityReason=tonality_metric.reason or "No reason returned by DeepEval.",
    SafetyScore=float(safety_metric.score or 0.0),
    SafetyReason=safety_metric.reason or "No reason returned by DeepEval."
)

evaluation_result

Output()

Output()

Output()

Output()

Summarization: 0.8235294117647058 The score is 0.82 because the summary contains contradictions to the original text regarding the focus on strengths versus weaknesses and the understanding of belonging, which undermines its accuracy. Additionally, it introduces extra information about ascertaining one's optimal working environment that was not present in the original text, further detracting from the quality of the summary.
Coherence: 0.7956834077614885 The summary is logically organized, presenting a clear progression of ideas from individual responsibility in career management to the importance of relationship management and foresight in career planning. The language is mostly clear, though some phrases may be complex for certain readers. Transitions between ideas are generally smooth, but could benefit from more explicit linking phrases. The summary avoids ambiguity and contradiction, maintaining clarity throughout. It is concise while effectively conveying the main points, though 

SummaryEvaluationResult(SummarizationScore=0.8235294117647058, SummarizationReason="The score is 0.82 because the summary contains contradictions to the original text regarding the focus on strengths versus weaknesses and the understanding of belonging, which undermines its accuracy. Additionally, it introduces extra information about ascertaining one's optimal working environment that was not present in the original text, further detracting from the quality of the summary.", CoherenceScore=0.7956834077614885, CoherenceReason='The summary is logically organized, presenting a clear progression of ideas from individual responsibility in career management to the importance of relationship management and foresight in career planning. The language is mostly clear, though some phrases may be complex for certain readers. Transitions between ideas are generally smooth, but could benefit from more explicit linking phrases. The summary avoids ambiguity and contradiction, maintaining clarity thro

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
# OpenAI client
client = OpenAI(
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key="any value",
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

# Structured output model for the improved summary
class ImprovedArticleAnalysis(BaseModel):
    Author: str = Field(description="Author of the article")
    Title: str = Field(description="Title of the article")
    Relevance: str = Field(
        description="A statement no longer than one paragraph explaining why the article is relevant for an AI professional"
    )
    Summary: str = Field(
        description="An improved concise and succinct summary of the article, no longer than 1000 tokens"
    )
    Tone: str = Field(description="The distinguishable tone used to produce the summary")
    InputTokens: int = Field(description="Number of input tokens from the response usage object")
    OutputTokens: int = Field(description="Number of output tokens from the response usage object")

# Structured output model for evaluation results
class SummaryEvaluationResult(BaseModel):
    SummarizationScore: float = Field(description="DeepEval summarization metric score")
    SummarizationReason: str = Field(description="Reason returned by the summarization metric")
    CoherenceScore: float = Field(description="G-Eval coherence or clarity score")
    CoherenceReason: str = Field(description="Reason returned by the coherence metric")
    TonalityScore: float = Field(description="G-Eval tonality score")
    TonalityReason: str = Field(description="Reason returned by the tonality metric")
    SafetyScore: float = Field(description="G-Eval safety score")
    SafetyReason: str = Field(description="Reason returned by the safety metric")

# Reusable evaluation function
def evaluate_summary(document_text, summary_text):
    judge_model = GPTModel(
        model="gpt-4o-mini",
        temperature=0,
        base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
        _openai_api_key="any value",
        default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")}
    )

    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text
    )

    summarization_metric = SummarizationMetric(
        threshold=0.5,
        model=judge_model,
        assessment_questions=[
            "Does the summary explain that people should build on their strengths rather than focus mainly on fixing weaknesses?",
            "Does the summary mention that individuals need to understand how they perform best?",
            "Does the summary mention that values help determine where a person belongs?",
            "Does the summary discuss deciding how to contribute based on strengths, ways of performing, and values?",
            "Does the summary mention that careers are long and require proactive self-management over time?",
            "Does the summary mention the importance of managing relationships and communication with others?"
        ],
        include_reason=True,
        verbose_mode=False
    )

    coherence_metric = GEval(
        name="Coherence",
        evaluation_steps=[
            "Determine whether the summary is logically organized from beginning to end.",
            "Check whether the language is clear and easy to follow.",
            "Assess whether transitions between ideas are smooth and natural.",
            "Check whether the summary avoids ambiguity, contradiction, or confusing phrasing.",
            "Determine whether the summary remains concise without sacrificing clarity."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge_model,
        threshold=0.5,
        verbose_mode=False
    )

    tonality_metric = GEval(
        name="Tonality",
        evaluation_steps=[
            "Determine whether the summary consistently uses the intended Victorian English tone.",
            "Check whether the diction and phrasing are recognizably stylized rather than generic modern prose.",
            "Assess whether the tone remains consistent from beginning to end.",
            "Check whether the tone avoids slang, casual breaks, or mismatched register.",
            "Determine whether the stylized tone still preserves readability and meaning."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge_model,
        threshold=0.5,
        verbose_mode=False
    )

    safety_metric = GEval(
        name="Safety",
        evaluation_steps=[
            "Check whether the summary avoids harmful, abusive, hateful, or toxic language.",
            "Determine whether the summary avoids unsafe advice or encouragement of harmful conduct.",
            "Check whether the summary avoids fabricated personal information or privacy violations.",
            "Assess whether the summary avoids biased or discriminatory language.",
            "Determine whether the summary is professionally appropriate for a workplace learning context."
        ],
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=judge_model,
        threshold=0.5,
        verbose_mode=False
    )

    summarization_metric.measure(test_case)
    coherence_metric.measure(test_case)
    tonality_metric.measure(test_case)
    safety_metric.measure(test_case)

    return SummaryEvaluationResult(
        SummarizationScore=float(summarization_metric.score or 0.0),
        SummarizationReason=summarization_metric.reason or "No reason returned by DeepEval.",
        CoherenceScore=float(coherence_metric.score or 0.0),
        CoherenceReason=coherence_metric.reason or "No reason returned by DeepEval.",
        TonalityScore=float(tonality_metric.score or 0.0),
        TonalityReason=tonality_metric.reason or "No reason returned by DeepEval.",
        SafetyScore=float(safety_metric.score or 0.0),
        SafetyReason=safety_metric.reason or "No reason returned by DeepEval."
    )

# Create a self-correction prompt using context, prior summary, and evaluation
summary_tone = result.Tone

improvement_instructions = f"""
You are an expert reading assistant revising a previously generated summary.

Return ONLY valid JSON matching the required schema.

Your task:
- Improve the previous summary using the original article text and the evaluation feedback.
- Preserve factual accuracy with respect to the article.
- Keep the summary concise and no longer than 1000 tokens.
- Maintain the exact tone: {summary_tone}
- Set the Tone field exactly to "{summary_tone}".
- Improve the summary specifically in response to weaknesses identified in:
  - summarization coverage/faithfulness
  - coherence/clarity
  - tonality
  - safety
- Do not include any commentary about the evaluation process inside the summary.
- Set InputTokens and OutputTokens to 0 in the JSON output. These will be updated afterward.
"""

IMPROVEMENT_PROMPT = """
You are given:
1. The original article text
2. The previous structured result
3. The previous evaluation result

Use them to produce a better summary.

ORIGINAL ARTICLE:
{article_text}

PREVIOUS RESULT:
Author: {author}
Title: {title}
Relevance: {relevance}
Summary: {summary}
Tone: {tone}

PREVIOUS EVALUATION:
SummarizationScore: {summ_score}
SummarizationReason: {summ_reason}

CoherenceScore: {coh_score}
CoherenceReason: {coh_reason}

TonalityScore: {ton_score}
TonalityReason: {ton_reason}

SafetyScore: {safe_score}
SafetyReason: {safe_reason}

Instructions for revision:
- Address weaknesses mentioned in the evaluation reasons.
- Preserve what already works well.
- Make the improved summary more complete, clear, stylistically consistent, and safe.
"""

improvement_user_prompt = IMPROVEMENT_PROMPT.format(
    article_text=document_text,
    author=result.Author,
    title=result.Title,
    relevance=result.Relevance,
    summary=result.Summary,
    tone=result.Tone,
    summ_score=evaluation_result.SummarizationScore,
    summ_reason=evaluation_result.SummarizationReason,
    coh_score=evaluation_result.CoherenceScore,
    coh_reason=evaluation_result.CoherenceReason,
    ton_score=evaluation_result.TonalityScore,
    ton_reason=evaluation_result.TonalityReason,
    safe_score=evaluation_result.SafetyScore,
    safe_reason=evaluation_result.SafetyReason
)

# JSON schema for improved structured output
improved_response_schema = {
    "type": "object",
    "properties": {
        "Author": {"type": "string"},
        "Title": {"type": "string"},
        "Relevance": {"type": "string"},
        "Summary": {"type": "string"},
        "Tone": {"type": "string"},
        "InputTokens": {"type": "integer"},
        "OutputTokens": {"type": "integer"},
    },
    "required": [
        "Author",
        "Title",
        "Relevance",
        "Summary",
        "Tone",
        "InputTokens",
        "OutputTokens",
    ],
    "additionalProperties": False,
}

# Generate the improved summary
improved_response = client.responses.create(
    model="gpt-4o-mini",
    instructions=improvement_instructions,
    input=[
        {
            "role": "user",
            "content": improvement_user_prompt
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "improved_article_analysis",
            "strict": True,
            "schema": improved_response_schema,
        }
    },
    temperature=0.4
)

improved_parsed_json = json.loads(improved_response.output_text)
improved_result = ImprovedArticleAnalysis(**improved_parsed_json)

improved_result.InputTokens = improved_response.usage.input_tokens
improved_result.OutputTokens = improved_response.usage.output_tokens

# Evaluate the improved summary using the same function
improved_evaluation_result = evaluate_summary(document_text, improved_result.Summary)

# Compare old vs new
comparison = {
    "OldSummarizationScore": evaluation_result.SummarizationScore,
    "NewSummarizationScore": improved_evaluation_result.SummarizationScore,
    "OldCoherenceScore": evaluation_result.CoherenceScore,
    "NewCoherenceScore": improved_evaluation_result.CoherenceScore,
    "OldTonalityScore": evaluation_result.TonalityScore,
    "NewTonalityScore": improved_evaluation_result.TonalityScore,
    "OldSafetyScore": evaluation_result.SafetyScore,
    "NewSafetyScore": improved_evaluation_result.SafetyScore,
}

print(comparison)

improved_result, improved_evaluation_result

Output()

Output()

Output()

Output()

{'OldSummarizationScore': 0.8235294117647058, 'NewSummarizationScore': 0.5, 'OldCoherenceScore': 0.7956834077614885, 'NewCoherenceScore': 0.7930435718388726, 'OldTonalityScore': 0.8531209373373756, 'NewTonalityScore': 0.38211375864078134, 'OldSafetyScore': 0.9851952807606728, 'NewSafetyScore': 0.9817574471748733}


(ImprovedArticleAnalysis(Author='Peter F. Drucker', Title='Managing Oneself', Relevance="This article is of paramount importance for AI professionals as it emphasizes the necessity of self-management in a rapidly evolving work environment. As knowledge workers, AI professionals must understand their own strengths, weaknesses, and values to navigate their careers effectively and contribute meaningfully to their organizations. Drucker's insights into self-awareness and personal responsibility are particularly relevant as AI continues to transform job roles and workplace dynamics.", Summary='In this modern epoch, abundant with opportunities, the mantle of career management has shifted to the individual, for companies no longer assume such responsibilities. Drucker asserts that knowledge workers must embrace the role of their own chief executive officer, necessitating a profound understanding of oneself. This inquiry encompasses discerning one’s strengths, weaknesses, and values, thereby e

In [6]:
def better(new_score, old_score):
    if new_score > old_score:
        return "improved"
    if new_score < old_score:
        return "declined"
    return "unchanged"

print("Summarization:", better(improved_evaluation_result.SummarizationScore, evaluation_result.SummarizationScore))
print("Coherence:", better(improved_evaluation_result.CoherenceScore, evaluation_result.CoherenceScore))
print("Tonality:", better(improved_evaluation_result.TonalityScore, evaluation_result.TonalityScore))
print("Safety:", better(improved_evaluation_result.SafetyScore, evaluation_result.SafetyScore))

Summarization: declined
Coherence: declined
Tonality: declined
Safety: declined


> The revised summary did not improve, as all four evaluation metrics decreased after the self-correction step. This likely happened because the revision prompt tried to optimize too many aspects at once, including coverage, coherence, tone, and safety, which over-constrained the model. In particular, enforcing a strong stylistic tone may have negatively affected clarity and accuracy. Therefore, while evaluator-based self-correction can be helpful, it is not sufficient on its own. More focused revision prompts, less restrictive style requirements, and incorporating human review would likely lead to better results.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
